In [49]:
%pip install tabulate

Note: you may need to restart the kernel to use updated packages.


In [50]:
from experiment_fraud import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
import matplotlib.pyplot as plt
from sklearn.neural_network import MLPClassifier
import pandas as pd
import numpy as np
import pickle as pkl
import time
from sklearn.metrics import precision_recall_curve, precision_recall_fscore_support

In [51]:
X, y = load_dataset()
train_x, test_x, train_y, test_y = train_test_split(X, y, test_size = 0.12)
train_x, val_x, train_y, val_y = train_test_split(train_x, train_y, test_size = 0.15)
train_x.shape, val_x.shape, test_x.shape

((213035, 28), (37595, 28), (34177, 28))

In [52]:
def find_p_r_combo_where_r_greater_than(x, y, model, recall_threshold: float):
    probs = model.predict_proba(x)[:, 1]
    precisions, recalls, thresholds = precision_recall_curve(y, probs)

    precisions = precisions[:-1]
    recalls = recalls[:-1]

    mask = recalls >= recall_threshold

    if np.any(mask):
        # Among the qualifying points, pick the one with highest precision
        best_idx = np.argmax(precisions[mask])
        threshold = thresholds[mask][best_idx]
        return threshold
    else:
        # No threshold can reach desired recall
        return None


In [53]:
architectures = [(16,), (32,), (64,), (128,), (256,), (512,), (64, 32), (128, 64), (256, 128)]
models = {}
for arch in architectures:
    mlp = MLPClassifier(arch)
    mlp.fit(train_x, train_y)
    models[arch] = mlp

In [54]:
rows = []
for arch, mlp in models.items():
    t = find_p_r_combo_where_r_greater_than(val_x, val_y, mlp, 0.75)
    test_pred = mlp.predict_proba(test_x)[:, 1] > t
    p, r, f, s = precision_recall_fscore_support(test_y, test_pred)

    rows.append({
        "Architecture": arch,
        "Precision": p[1],
        "Recall": r[1],
        "Fscore": f[1],
        "Support": s[1]
    })


    val_pred = mlp.predict_proba(val_x)[:, 1]
    p, r, t = precision_recall_curve(val_y, val_pred)

pr_results = pd.DataFrame(rows)
pr_results.to_csv("pr_results.csv")
pr_results[["Architecture", "Precision", "Recall"]].to_markdown('precision_recall.md')


In [55]:

log_reg = LogisticRegression()
log_reg.fit(train_x, 1 - train_y)
output = log_reg.predict_proba(train_x)

# Hard Coded to protect IP.
val = -4.125
t = 1 - 10 ** val
pd.Series(train_y[output[:, 1] > t]).value_counts()

Class
0    51806
Name: count, dtype: int64

In [56]:

# This indicates the number of data points that we've isolated in the non-fraudulent class.
mask = log_reg.predict_proba(train_x)[:, 1] > t # Don't bother predicting these again.
mask.sum()
    

np.int64(51806)

In [57]:
models

{(16,): MLPClassifier(hidden_layer_sizes=(16,)),
 (32,): MLPClassifier(hidden_layer_sizes=(32,)),
 (64,): MLPClassifier(hidden_layer_sizes=(64,)),
 (128,): MLPClassifier(hidden_layer_sizes=(128,)),
 (256,): MLPClassifier(hidden_layer_sizes=(256,)),
 (512,): MLPClassifier(hidden_layer_sizes=(512,)),
 (64, 32): MLPClassifier(hidden_layer_sizes=(64, 32)),
 (128, 64): MLPClassifier(hidden_layer_sizes=(128, 64)),
 (256, 128): MLPClassifier(hidden_layer_sizes=(256, 128))}

In [58]:
rows = []
for arch, mlp in models.items():

    # Pipeline predict is our new model, note how it 
    # uses logistic regression (just a hyperplane classifier)
    # to detect easy-to-classify points and avoids the expensive
    # model computation.
    def pipeline_predict(train_x, mlp, log_reg, t):
        output = np.zeros(len(train_x))
        mask = log_reg.predict_proba(train_x)[:, 1] > t
        start = time.time()
        output[~mask] = mlp.predict(train_x[~mask])
        end = time.time()

        output[mask] = 0
        return output, end - start

    # Warmup.
    for i in range(5):
        b_predictions = mlp.predict(X)
        pipeline_predict(X, mlp, log_reg, t)

    # Baseline
    start = time.time()
    baseline_times = []
    for i in range(5):
        s = time.time()
        b_predictions = mlp.predict(X)
        e = time.time()
        baseline_times.append(e - s)

    end = time.time()
    baseline_time = (end - start) / 5

    for i in range(5):
        b_predictions = mlp.predict(X)
        pipeline_predict(X, mlp, log_reg, t)

    # Experiment
    start = time.time()
    total_time = 0
    experiment_times = []
    for i in range(5):
        s = time.time()
        e_predictions, prediction_time = pipeline_predict(X, mlp, log_reg, t)
        e = time.time()
        total_time += prediction_time
        experiment_times.append(e - s)
    
    end = time.time()

    experimental_time = (end - start) / 5
    agreement = (b_predictions == e_predictions).mean()
    rows.append({"MLP Architecture": arch,
        "sklearn MLP time (s)": np.round(baseline_time, 6),
        "Gated MLP time (s)": np.round(experimental_time, 6),
        "adherence": agreement,
        "sklearn std": np.std(baseline_times),
        "gated std": np.std(experiment_times)
    })

results = pd.DataFrame(rows)


results.to_csv('out.csv')


In [60]:
for_report = results[["MLP Architecture", "sklearn MLP time (s)", "Gated MLP time (s)", 'adherence']]
for_report['Speedup'] = for_report['sklearn MLP time (s)'] / for_report["Gated MLP time (s)"]
for_report= for_report[["MLP Architecture", "sklearn MLP time (s)", "Gated MLP time (s)", "Speedup", 'adherence']]
for_report.to_markdown('output.md')

/var/folders/gk/4qpy_m5j67d9h1h0g9j07wzr0000gn/T/ipykernel_57905/4236618802.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  for_report['Speedup'] = for_report['sklearn MLP time (s)'] / for_report["Gated MLP time (s)"]


In [61]:
for_report

,MLP Architecture,sklearn MLP time (s),Gated MLP time (s),Speedup,adherence
0,"(16,)",0.018979,0.032594,0.582285,1.0
1,"(32,)",0.026658,0.035776,0.745136,1.0
2,"(64,)",0.051625,0.044983,1.147656,1.0
3,"(128,)",0.084832,0.084571,1.003086,1.0
4,"(256,)",0.162389,0.137427,1.181638,1.0
5,"(512,)",0.771465,0.285715,2.700121,1.0
6,"(64, 32)",0.065221,0.062808,1.038419,1.0
7,"(128, 64)",0.149000,0.123228,1.209141,1.0
8,"(256, 128)",0.337900,0.263982,1.280012,1.0
